# Database Secrets Engine: Lumen RO/RW

`./demo.sh` includes creating the catalog, tables and roles. It reuses MySQL/KMIP from [notebook 2](2_vault-demo-kmip-mysql.ipynb). The application's connection and roles live in the Vault namespace **verify**, not in the root namespace.

| Role | SQL privileges | TTL / max |
|---|---|---|
| `chatbot-readonly` | SELECT on `scripting.*` | 120 s / 300 s |
| `chatbot-readwrite` | SELECT, INSERT, UPDATE, DELETE on `scripting.*` | 120 s / 300 s |

Vault manages the `vault_chatbot_admin` account and rotates its password when creating the connection. The application requests credentials with OBO and RAR; the human profile, the agent ceiling and the RAR must all allow the path.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/01-database.sh


In [ ]:
%%bash
source verify/scripts/common.sh
for role in chatbot-readonly chatbot-readwrite; do
  vault read -format=json "database/roles/$role" | jq --arg role "$role" '{role:$role,db_name:.data.db_name,default_ttl:.data.default_ttl,max_ttl:.data.max_ttl,creation_statements:.data.creation_statements}'
done


## Verification

After the full deployment, the check validates SELECT, allows INSERT with RW, denies INSERT with RO and rolls back. It also checks Tavily and the bootstrap token limits. Per-user OBO tests require that user's real login in Verify.


In [ ]:
%%bash
set -euo pipefail
bash chatbot/scripts/test-infrastructure.sh


The old `database/config/mysql` connections and the `payments-app` static role in the root namespace belong to another exercise; if they exist they are preserved, but Lumen does not use them. The `5_Payments_App.ipynb` application is not included; it is not part of this workspace.

[Notebook 8](8_Lumen_Protected_Applications.ipynb) adds Transit/Transform. The supervisor uses SQL RO and separate permissions to reveal data; the writer uses SQL RW and can protect data, but not reveal it.
